# Creacion de Network Dataset

In [2]:
import arcpy

arcpy.env.overwriteOutput = True

gdb = r"C:\EsriTraining\TFM\GDB\GDB_Principal.gdb"
feature_dataset = gdb + r"\Red_Viaria"
carreteras = feature_dataset + r"\Carreteras"

# ============================================================
# 1. CALCULAR VELOCIDADES (maxspeed = 0 → asignar por fclass)
# ============================================================
print("Calculando velocidades...")

velocidades = {
    "motorway":       120, "motorway_link":  120,
    "trunk":           90, "trunk_link":      90,
    "primary":         90, "primary_link":    90,
    "secondary":       50, "secondary_link":  50,
    "tertiary":        50, "tertiary_link":   50,
    "residential":     30, "service":         30,
    "unclassified":    30, "track":           30,
    "track_grade1":    30, "track_grade2":    30,
    "track_grade3":    30, "track_grade4":    30,
    "track_grade5":    30, "busway":          30,
    "living_street":   20, "pedestrian":      20,
}

with arcpy.da.UpdateCursor(carreteras, ["fclass", "maxspeed"]) as cursor:
    for row in cursor:
        if row[1] == 0 or row[1] is None:
            row[1] = velocidades.get(row[0], 30)
            cursor.updateRow(row)

print("✅ Velocidades calculadas")

# ============================================================
# 2. AÑADIR CAMPOS NUEVOS
# ============================================================
print("Añadiendo campos...")

campos_nuevos = [
    ("Kilometros",          "DOUBLE"),
    ("Minutos_Conduciendo", "DOUBLE"),
    ("Minutos_Andando",     "DOUBLE"),
    ("Jerarquia",           "LONG"),
]

campos_existentes = [f.name for f in arcpy.ListFields(carreteras)]
for nombre, tipo in campos_nuevos:
    if nombre not in campos_existentes:
        arcpy.management.AddField(carreteras, nombre, tipo)
        print(f"  + Campo añadido: {nombre}")
    else:
        print(f"  · Ya existe: {nombre}")

print("✅ Campos listos")

# ============================================================
# 3. CALCULAR KILOMETROS
# ============================================================
print("Calculando kilómetros...")
arcpy.management.CalculateGeometryAttributes(
    carreteras,
    [["Kilometros", "LENGTH_GEODESIC"]],
    "KILOMETERS"
)
print("✅ Kilómetros calculados")

# ============================================================
# 4. CALCULAR TIEMPOS
# ============================================================
print("Calculando tiempos...")
arcpy.management.CalculateField(
    carreteras, "Minutos_Conduciendo",
    "(!Kilometros! * 60 / !maxspeed!) * 1.5",
    "PYTHON3"
)
arcpy.management.CalculateField(
    carreteras, "Minutos_Andando",
    "!Kilometros! / 6 * 60",
    "PYTHON3"
)
print("✅ Tiempos calculados")

# ============================================================
# 5. CALCULAR JERARQUIA
# ============================================================
print("Calculando jerarquía...")

jerarquia_map = {
    1: ["motorway", "motorway_link", "primary", "primary_link", "trunk", "trunk_link"],
    2: ["secondary", "secondary_link", "tertiary", "tertiary_link"],
    3: ["residential", "service", "living_street", "busway",        # ← busway movido aquí
        "track", "track_grade1", "track_grade2", "track_grade3",
        "track_grade4", "track_grade5", "unclassified"],
    4: ["pedestrian", "cycleway", "footway", "path", "bridleway", "steps"],  # ← busway quitado
}

fclass_to_jerarquia = {}
for nivel, fclasses in jerarquia_map.items():
    for fc in fclasses:
        fclass_to_jerarquia[fc] = nivel

with arcpy.da.UpdateCursor(carreteras, ["fclass", "Jerarquia"]) as cursor:
    for row in cursor:
        row[1] = fclass_to_jerarquia.get(row[0], 3)  # default 3 si fclass desconocido
        cursor.updateRow(row)

print("✅ Jerarquía calculada")

# ============================================================
# 6. DIVIDIR EN SUPERFICIE / PASOS_ELEVADOS / PASOS_SUBTERRANEOS
# ============================================================
print("Dividiendo en subcapas...")

superficie      = feature_dataset + r"\Superficie"
pasos_elevados  = feature_dataset + r"\Pasos_Elevados"
pasos_sub       = feature_dataset + r"\Pasos_Subterraneos"

# Eliminar si existen
for capa in [superficie, pasos_elevados, pasos_sub]:
    if arcpy.Exists(capa):
        arcpy.management.Delete(capa)

# Exportar cada grupo
arcpy.conversion.ExportFeatures(carreteras, pasos_elevados, "bridge = 'T'")
print("✅ Pasos_Elevados creado")

arcpy.conversion.ExportFeatures(carreteras, pasos_sub, "tunnel = 'T'")
print("✅ Pasos_Subterraneos creado")

arcpy.conversion.ExportFeatures(carreteras, superficie, "bridge = 'F' AND tunnel = 'F'")
print("✅ Superficie creada")

# ============================================================
# 7. INTEGRATE (conectar vértices)
# ============================================================
print("Ejecutando Integrate...")
arcpy.management.Integrate(
    in_features=[superficie, pasos_elevados, pasos_sub],
    cluster_tolerance="0.001 Meters"
)
print("✅ Integrate completado")

# ============================================================
# 8. CREAR NETWORK DATASET
# ============================================================
print("Creando Network Dataset...")

nd_madrid = feature_dataset + r"\ND_Madrid_Notebook"
if arcpy.Exists(nd_madrid):
    arcpy.management.Delete(nd_madrid)

arcpy.na.CreateNetworkDataset(
    feature_dataset=feature_dataset,
    out_name="ND_Madrid_Notebook",
    source_feature_class_names=["Superficie", "Pasos_Elevados", "Pasos_Subterraneos"],
    elevation_model="ELEVATION_FIELDS"
)
print("✅ Network Dataset creado")

# ============================================================
# 9. BUILD
# ============================================================
# print("Construyendo red...")
# arcpy.na.BuildNetwork(nd_madrid)
# print("✅ Red construida")

print("\n✅ TODO COMPLETADO")
print("⚠️  Ahora configura en ArcGIS Pro → ND_Madrid_Notebook → Properties:")
print("   Source Settings → Group Connectivity:")
print("     - Superficie       → Any Vertex")
print("     - Pasos_Elevados   → Endpoint")
print("     - Pasos_Subterraneos → Endpoint")
print("   Travel Attributes → añadir costes, restricciones y travel modes")
print("   Luego Build de nuevo")

Calculando velocidades...
✅ Velocidades calculadas
Añadiendo campos...
  · Ya existe: Kilometros
  · Ya existe: Minutos_Conduciendo
  · Ya existe: Minutos_Andando
  · Ya existe: Jerarquia
✅ Campos listos
Calculando kilómetros...
✅ Kilómetros calculados
Calculando tiempos...
✅ Tiempos calculados
Calculando jerarquía...
✅ Jerarquía calculada
Dividiendo en subcapas...
✅ Pasos_Elevados creado
✅ Pasos_Subterraneos creado
✅ Superficie creada
Ejecutando Integrate...
✅ Integrate completado
Creando Network Dataset...
✅ Network Dataset creado
Construyendo red...
✅ Red construida

✅ TODO COMPLETADO
⚠️  Ahora configura en ArcGIS Pro → ND_Madrid → Properties:
   Source Settings → Group Connectivity:
     - Superficie       → Any Vertex
     - Pasos_Elevados   → Endpoint
     - Pasos_Subterraneos → Endpoint
   Travel Attributes → añadir costes, restricciones y travel modes
   Luego Build de nuevo
